# Objetivo especifico 1 - Resultado esperado 1.1

Desarrollo y entrenamiento de un modelo de difusion condicionado para generar imagenes sinteticas de patologias retinales de baja incidencia sobre RFMiD 2.0.

Arquitectura: modelo de difusion latente (LDM) tipo Medfusion (Pandey et al., 2025). Autoencoder variacional preentrenado congelado mas U-Net con atencion cruzada entrenada en el espacio latente, condicionada por un embedding multi-etiqueta de las clases foco ME, MHL, MYA y CWS con Classifier-Free Guidance.

El notebook detecta el entorno (local, Colab, Kaggle) y ajusta rutas, descargas y checkpoints. En local activa `SMOKE_TEST` para validar el pipeline en una sola epoca.

## 1. Deteccion de entorno

In [ ]:
import os
from pathlib import Path

IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ
try:
    import google.colab  # noqa: F401
    IS_COLAB = True
except ImportError:
    IS_COLAB = False
IS_LOCAL = not (IS_KAGGLE or IS_COLAB)

ENV = 'kaggle' if IS_KAGGLE else 'colab' if IS_COLAB else 'local'
print('entorno detectado:', ENV)

## 2. Instalacion de dependencias (solo Colab y Kaggle)

En local se asumen instaladas via `requirements.txt` dentro del `.venv`.

In [ ]:
if IS_COLAB or IS_KAGGLE:
    import subprocess
    subprocess.run([
        'pip', 'install', '-q',
        'diffusers==0.27.2',
        'accelerate==0.30.1',
        'transformers==4.40.2',
        'safetensors==0.4.3',
    ], check=True)

## 3. Imports y semilla

In [ ]:
import json
import random
import zipfile
import urllib.request
import shutil

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import matplotlib.pyplot as plt

from diffusers import AutoencoderKL, UNet2DConditionModel, DDPMScheduler, DPMSolverMultistepScheduler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, '| gpu:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## 4. Configuracion por entorno

`SMOKE_TEST` fuerza una epoca, batch reducido y una submuestra pequena. Se activa por defecto en local para validar el pipeline en CPU.

In [ ]:
REPO_URL = 'https://github.com/benzCase/fuzzy-diffusion'
RELEASE_TAG = 'rfmid2-v1'
ASSETS = ['Training_set.zip', 'Validation_set.zip', 'Test_set.zip', 'RFMiD_2_Training_labels.csv']

if IS_LOCAL:
    NOTEBOOK_DIR = Path.cwd()
    REPO_ROOT = NOTEBOOK_DIR
    for _ in range(5):
        if (REPO_ROOT / 'data' / 'raw').exists():
            break
        REPO_ROOT = REPO_ROOT.parent
    DATA_ROOT = REPO_ROOT / 'data' / 'raw'
    OUT_DIR = REPO_ROOT / 'diffusion-model' / 'obj-1' / 're_1-1' / 'outputs'
elif IS_COLAB:
    DATA_ROOT = Path('/content/data/raw')
    OUT_DIR = Path('/content/outputs/re_1_1')
else:
    KAGGLE_INPUT = Path('/kaggle/input')
    candidate = next(iter(KAGGLE_INPUT.glob('*/data/raw')), None) if KAGGLE_INPUT.exists() else None
    DATA_ROOT = candidate if candidate is not None else Path('/kaggle/working/data/raw')
    OUT_DIR = Path('/kaggle/working/re_1_1')

OUT_DIR.mkdir(parents=True, exist_ok=True)
print('DATA_ROOT:', DATA_ROOT)
print('OUT_DIR :', OUT_DIR)

SMOKE_TEST = IS_LOCAL
RESUME_FROM = OUT_DIR / 'last'

CFG = {
    'image_size': 256,
    'latent_size': 32,
    'latent_channels': 4,
    'focus_classes': ['ME', 'MHL', 'MYA', 'CWS'],
    'batch_size': 16,
    'lr': 1e-4,
    'epochs': 400,
    'p_uncond': 0.1,
    'guidance_scale': 4.0,
    'num_train_timesteps': 1000,
    'num_infer_timesteps': 50,
    'vae_id': 'stabilityai/sd-vae-ft-mse',
    'vae_scale': 0.18215,
    'smoke_subset': 16,
}

if SMOKE_TEST:
    CFG['epochs'] = 1
    CFG['batch_size'] = 4
    CFG['num_infer_timesteps'] = 10
    print('SMOKE_TEST activo: 1 epoca, batch 4, subset', CFG['smoke_subset'])

## 5. Descarga de datos en Colab y Kaggle

Descarga los assets del release publico y extrae los tres zips. En local no hace nada porque la data ya esta en disco.

In [ ]:
def download(url, dest):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        print('ya presente:', dest.name)
        return
    print('descargando', dest.name)
    urllib.request.urlretrieve(url, dest)

if not IS_LOCAL and not any(DATA_ROOT.glob('train/*.jpg')):
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    for name in ASSETS:
        url = f'{REPO_URL}/releases/download/{RELEASE_TAG}/{name}'
        download(url, DATA_ROOT / name)
    for zname, sub in [('Training_set.zip', 'train'), ('Validation_set.zip', 'val'), ('Test_set.zip', 'test')]:
        zpath = DATA_ROOT / zname
        if zpath.exists() and not (DATA_ROOT / sub).exists():
            print('extrayendo', zname)
            with zipfile.ZipFile(zpath) as zf:
                zf.extractall(DATA_ROOT)

for sub in ['train', 'val', 'test']:
    p = DATA_ROOT / sub
    print(f'{sub}: exists={p.exists()}, files={len(list(p.glob("*"))) if p.exists() else 0}')

## 6. Carga y limpieza de los tres archivos de etiquetas

Se cargan los CSV de train, validacion y test de RFMiD 2.0 preservando los splits nativos del dataset (opcion A: los mismos splits que propone Panchal et al. 2023). Se unifica ME con CME cuando corresponda y se descartan filas sin imagen asociada.

In [ ]:
def clean_columns(df):
    df.columns = [c.strip().replace('\ufeff', '') for c in df.columns]
    df = df.loc[:, ~df.columns.str.contains('^Unnamed', na=False)]
    df = df.loc[:, [c for c in df.columns if c != '']]
    return df

def load_split_csv(split_name):
    if split_name == 'train':
        candidates = [DATA_ROOT / 'RFMiD_2_Training_labels.csv',
                      DATA_ROOT / 'train' / 'RFMiD_2_Training_labels.csv']
    elif split_name == 'val':
        candidates = [DATA_ROOT / 'RFMiD_2_Validation_labels.csv',
                      DATA_ROOT / 'val' / 'RFMiD_2_Validation_labels.csv']
    else:
        candidates = [DATA_ROOT / 'RFMiD_2_Testing_labels.csv',
                      DATA_ROOT / 'test' / 'RFMiD_2_Testing_labels.csv']
    path = next((p for p in candidates if p.exists()), None)
    if path is None:
        raise FileNotFoundError(f'no se encontro CSV para split {split_name}: {candidates}')
    df = clean_columns(pd.read_csv(path, encoding='latin-1'))
    if 'CME' in df.columns and 'ME' in df.columns:
        df['ME'] = ((df['ME'] == 1) | (df['CME'] == 1)).astype(int)
        df = df.drop(columns=['CME'])
    if 'WNL' in df.columns:
        df = df.drop(columns=['WNL'])
    df['split'] = split_name
    return df

def attach_paths(df, folder):
    def find(id_):
        for ext in ['.jpg', '.png', '.jpeg', '.JPG', '.PNG', '.JPEG']:
            p = DATA_ROOT / folder / f'{id_}{ext}'
            if p.exists():
                return str(p)
        return None
    df = df.copy()
    df['path'] = df['ID'].apply(find)
    return df

train_raw = attach_paths(load_split_csv('train'), 'train')
val_raw = attach_paths(load_split_csv('val'), 'val')
test_raw = attach_paths(load_split_csv('test'), 'test')

for name, part in [('train', train_raw), ('val', val_raw), ('test', test_raw)]:
    missing = part['path'].isna().sum()
    print(f'{name}: total={len(part)}, sin_imagen={missing}')

train_df = train_raw.dropna(subset=['path']).reset_index(drop=True)
val_df = val_raw.dropna(subset=['path']).reset_index(drop=True)
test_df = test_raw.dropna(subset=['path']).reset_index(drop=True)

for c in CFG['focus_classes']:
    for part in [train_df, val_df, test_df]:
        if c not in part.columns:
            part[c] = 0

print('\ntamanos finales:', len(train_df), len(val_df), len(test_df))
for c in CFG['focus_classes']:
    tr = int(train_df[c].sum())
    va = int(val_df[c].sum())
    te = int(test_df[c].sum())
    print(f'  {c}: train={tr}, val={va}, test={te}')

## 7. Submuestreo cuando SMOKE_TEST esta activo

In [ ]:
if SMOKE_TEST:
    n_tr = min(CFG['smoke_subset'], len(train_df))
    n_va = min(8, len(val_df))
    train_df = train_df.sample(n=n_tr, random_state=SEED).reset_index(drop=True)
    val_df = val_df.sample(n=n_va, random_state=SEED).reset_index(drop=True)
    print('SMOKE_TEST: train', len(train_df), 'val', len(val_df))

## 8. Dataset de imagenes y transformaciones

In [ ]:
class RetinaDataset(Dataset):
    def __init__(self, df, classes, size=256):
        self.df = df.reset_index(drop=True)
        self.classes = classes
        self.tf = transforms.Compose([
            transforms.Resize((size, size), interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3),
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(row['path']).convert('RGB')
        x = self.tf(img)
        y = torch.tensor([float(row[c]) for c in self.classes], dtype=torch.float32)
        return x, y

## 9. VAE preentrenado congelado

In [ ]:
vae = AutoencoderKL.from_pretrained(CFG['vae_id']).to(device).eval()
for p in vae.parameters():
    p.requires_grad = False
print('vae params:', sum(p.numel() for p in vae.parameters()))

## 10. Precomputo de latentes

Codifica train con la version original y una version volteada horizontalmente cuando no es SMOKE_TEST. Codifica val sin aumentacion. Esto saca el VAE del bucle principal de entrenamiento.

In [ ]:
@torch.no_grad()
def encode_split(df, augment=False):
    ds = RetinaDataset(df, CFG['focus_classes'], size=CFG['image_size'])
    zs, ys = [], []
    for i in range(len(ds)):
        x, y = ds[i]
        variants = [x] + ([torch.flip(x, dims=[-1])] if augment else [])
        for v in variants:
            z = vae.encode(v.unsqueeze(0).to(device)).latent_dist.sample() * CFG['vae_scale']
            zs.append(z.cpu())
            ys.append(y)
    return torch.cat(zs, dim=0), torch.stack(ys, dim=0)

train_z, train_y = encode_split(train_df, augment=not SMOKE_TEST)
val_z, val_y = encode_split(val_df, augment=False)
print('train latents:', tuple(train_z.shape), 'val latents:', tuple(val_z.shape))

In [ ]:
class LatentDataset(Dataset):
    def __init__(self, z, y):
        self.z = z
        self.y = y

    def __len__(self):
        return len(self.z)

    def __getitem__(self, i):
        return self.z[i], self.y[i]

num_workers = 0 if IS_LOCAL else 2
train_loader = DataLoader(LatentDataset(train_z, train_y), batch_size=CFG['batch_size'], shuffle=True, num_workers=num_workers, drop_last=True)
val_loader = DataLoader(LatentDataset(val_z, val_y), batch_size=CFG['batch_size'], shuffle=False, num_workers=num_workers)

## 11. Codificador de condicion multi-etiqueta

Cada clase foco tiene un token aprendido. La condicion final es la suma ponderada por el vector multi-hot proyectada a una secuencia de tokens que la U-Net consume via atencion cruzada. Un token nulo aprendido se usa para Classifier-Free Guidance.

In [ ]:
COND_DIM = 128
COND_SEQ = 4

class ClassEncoder(nn.Module):
    def __init__(self, num_classes, dim, seq_len):
        super().__init__()
        self.dim = dim
        self.seq_len = seq_len
        self.class_tokens = nn.Parameter(torch.randn(num_classes, dim) * 0.02)
        self.null_seq = nn.Parameter(torch.randn(seq_len, dim) * 0.02)
        self.proj = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.SiLU(),
            nn.Linear(dim * 2, dim * seq_len),
        )

    def forward(self, y):
        pooled = (y.unsqueeze(-1) * self.class_tokens.unsqueeze(0)).sum(dim=1)
        return self.proj(pooled).view(y.size(0), self.seq_len, self.dim)

    def null(self, batch_size):
        return self.null_seq.unsqueeze(0).expand(batch_size, -1, -1).contiguous()

class_encoder = ClassEncoder(num_classes=len(CFG['focus_classes']), dim=COND_DIM, seq_len=COND_SEQ).to(device)

## 12. U-Net latente con atencion cruzada

In [ ]:
unet = UNet2DConditionModel(
    sample_size=CFG['latent_size'],
    in_channels=CFG['latent_channels'],
    out_channels=CFG['latent_channels'],
    layers_per_block=2,
    block_out_channels=(128, 256, 384, 384),
    down_block_types=('CrossAttnDownBlock2D', 'CrossAttnDownBlock2D', 'CrossAttnDownBlock2D', 'DownBlock2D'),
    up_block_types=('UpBlock2D', 'CrossAttnUpBlock2D', 'CrossAttnUpBlock2D', 'CrossAttnUpBlock2D'),
    cross_attention_dim=COND_DIM,
    attention_head_dim=8,
).to(device)

print('unet params:', sum(p.numel() for p in unet.parameters()))
print('class encoder params:', sum(p.numel() for p in class_encoder.parameters()))

train_scheduler = DDPMScheduler(num_train_timesteps=CFG['num_train_timesteps'], beta_schedule='linear', prediction_type='epsilon')

## 13. Reanudacion desde checkpoint (opcional)

Si existe `OUT_DIR/last/` con pesos y estado del optimizador el entrenamiento continua desde la epoca guardada. Sirve para migrar entre Colab y Kaggle: se descarga la carpeta `last/` del entorno anterior y se sube al nuevo (como Dataset en Kaggle, como carpeta en `/content/` en Colab). En SMOKE_TEST no se reanuda.

In [ ]:
params = list(unet.parameters()) + list(class_encoder.parameters())
optimizer = torch.optim.AdamW(params, lr=CFG['lr'], weight_decay=1e-6)
scaler = torch.cuda.amp.GradScaler(enabled=(device == 'cuda'))

start_epoch = 1
best_val = float('inf')
history = {'epoch': [], 'train_loss': [], 'val_loss': []}

if RESUME_FROM.exists() and (RESUME_FROM / 'unet').exists() and not SMOKE_TEST:
    print('reanudando desde', RESUME_FROM)
    unet = UNet2DConditionModel.from_pretrained(RESUME_FROM / 'unet').to(device)
    class_encoder.load_state_dict(torch.load(RESUME_FROM / 'class_encoder.pt', map_location=device))
    optimizer = torch.optim.AdamW(list(unet.parameters()) + list(class_encoder.parameters()), lr=CFG['lr'], weight_decay=1e-6)
    state = torch.load(RESUME_FROM / 'train_state.pt', map_location=device)
    optimizer.load_state_dict(state['optimizer'])
    start_epoch = state['epoch'] + 1
    best_val = state['best_val']
    history = state['history']
    params = list(unet.parameters()) + list(class_encoder.parameters())
    print('reanuda en epoca', start_epoch, 'best_val previo', best_val)

## 14. Bucle de entrenamiento con Classifier-Free Guidance

En cada paso el vector de condicion se sustituye por el token nulo con probabilidad `p_uncond`. Al final de cada epoca se guardan `last/` (siempre) y `best/` (si mejora val). Solo se conservan esas dos carpetas para no llenar los 3 GB del Drive gratuito.

In [ ]:
def save_checkpoint(tag, epoch):
    d = OUT_DIR / tag
    if d.exists():
        shutil.rmtree(d)
    d.mkdir(parents=True, exist_ok=True)
    unet.save_pretrained(d / 'unet')
    torch.save(class_encoder.state_dict(), d / 'class_encoder.pt')
    torch.save({
        'epoch': epoch,
        'optimizer': optimizer.state_dict(),
        'best_val': best_val,
        'history': history,
        'cfg': CFG,
    }, d / 'train_state.pt')

for epoch in range(start_epoch, CFG['epochs'] + 1):
    unet.train()
    class_encoder.train()
    losses = []
    for z, y in train_loader:
        z = z.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        cond = class_encoder(y)
        null = class_encoder.null(z.size(0))
        drop = (torch.rand(z.size(0), device=device) < CFG['p_uncond']).view(-1, 1, 1)
        ctx = torch.where(drop, null, cond)

        noise = torch.randn_like(z)
        t = torch.randint(0, CFG['num_train_timesteps'], (z.size(0),), device=device, dtype=torch.long)
        zn = train_scheduler.add_noise(z, noise, t)

        optimizer.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast(enabled=(device == 'cuda')):
            pred = unet(zn, t, encoder_hidden_states=ctx).sample
            loss = F.mse_loss(pred, noise)
        if device == 'cuda':
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            optimizer.step()
        losses.append(loss.item())

    train_loss = float(np.mean(losses))

    unet.eval()
    class_encoder.eval()
    v_losses = []
    with torch.no_grad():
        for z, y in val_loader:
            z = z.to(device)
            y = y.to(device)
            cond = class_encoder(y)
            noise = torch.randn_like(z)
            t = torch.randint(0, CFG['num_train_timesteps'], (z.size(0),), device=device, dtype=torch.long)
            zn = train_scheduler.add_noise(z, noise, t)
            with torch.cuda.amp.autocast(enabled=(device == 'cuda')):
                pred = unet(zn, t, encoder_hidden_states=cond).sample
                v_losses.append(F.mse_loss(pred, noise).item())
    val_loss = float(np.mean(v_losses)) if v_losses else float('nan')

    history['epoch'].append(epoch)
    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    print(f'epoch {epoch:03d}  train {train_loss:.4f}  val {val_loss:.4f}')

    save_checkpoint('last', epoch)
    if val_loss < best_val:
        best_val = val_loss
        save_checkpoint('best', epoch)

    with open(OUT_DIR / 'history.json', 'w') as f:
        json.dump(history, f)

## 15. Curvas de perdida

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(history['epoch'], history['train_loss'], label='train')
plt.plot(history['epoch'], history['val_loss'], label='val')
plt.xlabel('epoca')
plt.ylabel('MSE de ruido')
plt.title('Curvas de perdida - LDM condicionado')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUT_DIR / 'loss_curves.png', dpi=120)
plt.show()

## 16. Funcion de muestreo condicional con CFG

In [ ]:
@torch.no_grad()
def sample(labels, num_steps=None, guidance_scale=None, seed=None):
    num_steps = num_steps or CFG['num_infer_timesteps']
    guidance_scale = guidance_scale if guidance_scale is not None else CFG['guidance_scale']

    infer_scheduler = DPMSolverMultistepScheduler.from_config(train_scheduler.config)
    infer_scheduler.set_timesteps(num_steps)

    B = labels.size(0)
    gen = torch.Generator(device=device)
    if seed is not None:
        gen.manual_seed(int(seed))
    z = torch.randn(B, CFG['latent_channels'], CFG['latent_size'], CFG['latent_size'], device=device, generator=gen)

    unet.eval()
    class_encoder.eval()
    cond = class_encoder(labels.to(device))
    uncond = class_encoder.null(B)
    ctx = torch.cat([uncond, cond], dim=0)

    for t in infer_scheduler.timesteps:
        zz = torch.cat([z, z], dim=0)
        with torch.cuda.amp.autocast(enabled=(device == 'cuda')):
            pred = unet(zz, t, encoder_hidden_states=ctx).sample
        p_uncond, p_cond = pred.chunk(2)
        pred = p_uncond + guidance_scale * (p_cond - p_uncond)
        z = infer_scheduler.step(pred, t, z).prev_sample

    z = z / CFG['vae_scale']
    with torch.cuda.amp.autocast(enabled=(device == 'cuda')):
        imgs = vae.decode(z).sample
    imgs = (imgs.float().clamp(-1, 1) + 1) / 2
    return imgs.cpu()

## 17. Grilla de muestras por clase foco

In [ ]:
N_PER_CLASS = 2 if SMOKE_TEST else 4

def one_hot(idx, num=len(CFG['focus_classes'])):
    v = torch.zeros(num)
    v[idx] = 1.0
    return v

samples_dir = OUT_DIR / 'samples'
samples_dir.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(len(CFG['focus_classes']), N_PER_CLASS, figsize=(4 * N_PER_CLASS, 4 * len(CFG['focus_classes'])))
for r, cname in enumerate(CFG['focus_classes']):
    y = torch.stack([one_hot(r) for _ in range(N_PER_CLASS)])
    imgs = sample(y, seed=SEED + r)
    for c in range(N_PER_CLASS):
        arr = imgs[c].permute(1, 2, 0).numpy()
        ax = axes[r, c] if N_PER_CLASS > 1 else axes[r]
        ax.imshow(arr)
        ax.axis('off')
        if c == 0:
            ax.set_title(cname, loc='left', fontsize=14)
        Image.fromarray((arr * 255).astype(np.uint8)).save(samples_dir / f'{cname}_{c}.png')

plt.tight_layout()
plt.savefig(OUT_DIR / 'samples_grid.png', dpi=120)
plt.show()

## 18. Reporte de entrenamiento

In [ ]:
report = {
    'resultado_esperado': 'RE 1.1',
    'entorno': ENV,
    'smoke_test': SMOKE_TEST,
    'arquitectura': 'LDM tipo Medfusion - VAE SD-VAE-ft-mse congelado y UNet2DConditionModel condicionada por clase con CFG',
    'referencia_estado_del_arte': 'Pandey et al. (2025) Medfusion; refuerzo con Lopukhova et al. (2026)',
    'vae_id': CFG['vae_id'],
    'image_size': CFG['image_size'],
    'latent_shape': [CFG['latent_channels'], CFG['latent_size'], CFG['latent_size']],
    'focus_classes': CFG['focus_classes'],
    'splits': {'train': len(train_df), 'val': len(val_df), 'test': len(test_df)},
    'n_train_latents': int(train_z.size(0)),
    'batch_size': CFG['batch_size'],
    'epochs': CFG['epochs'],
    'lr': CFG['lr'],
    'p_uncond': CFG['p_uncond'],
    'guidance_scale': CFG['guidance_scale'],
    'num_train_timesteps': CFG['num_train_timesteps'],
    'num_infer_timesteps': CFG['num_infer_timesteps'],
    'unet_params': sum(p.numel() for p in unet.parameters()),
    'class_encoder_params': sum(p.numel() for p in class_encoder.parameters()),
    'final_train_loss': history['train_loss'][-1] if history['train_loss'] else None,
    'final_val_loss': history['val_loss'][-1] if history['val_loss'] else None,
    'best_val_loss': best_val,
    'seed': SEED,
    'device': str(device),
}

with open(OUT_DIR / 'training_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2, ensure_ascii=False))